
<img src="https://reqlut2.s3.sa-east-1.amazonaws.com/reqlut-images/duoc/logo.png?v=87.8" width="180px"/>


In [1]:
import sys
import os

"""
En caso de que Python no encuentre en la ruta los otros directorios,
ejecutar esta configuración
"""

sys.path.append(os.path.abspath(".."))

# Hiperparámetros modelo Random Forest

In [2]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import randint
import pandas as pd
import numpy as np
from src.carga import cargar_csv
import json

## Predicción de la depresión en estudiantes

Este cuaderno muestra un flujo de trabajo de aprendizaje automático para predecir la depresión en estudiantes basándose en diversas características. El proceso incluye la carga de datos, la exploración inicial, la división de datos, el entrenamiento del modelo de referencia, el ajuste de hiperparámetros mediante `GridSearchCV` y `RandomizedSearchCV`, y la evaluación del modelo final.

### 1. Carga de datos y exploración inicial

En primer lugar, cargamos el conjunto de datos preprocesado sobre la depresión en estudiantes y realizamos una inspección inicial de su estructura y contenido.


In [3]:
df = cargar_csv("..\data\processed\Student_Depression_Dataset_codificado.csv")
df.head()

,cat__Gender_Male,cat__Sleep Duration_7-8 hours,cat__Sleep Duration_Less than 5 hours,cat__Sleep Duration_More than 8 hours,cat__Dietary Habits_Moderate,cat__Dietary Habits_Unhealthy,cat__Degree_B.Com,cat__Degree_B.Ed,cat__Degree_B.Tech,cat__Degree_BCA,...,cat__Degree_Other,bin__Have you ever had suicidal thoughts ?,bin__Family History of Mental Illness,num__Age,num__Academic Pressure,num__CGPA,num__Study Satisfaction,num__Work/Study Hours,num__Financial Stress,remainder__Depression
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.476593,1.345273,0.895400,-0.694638,-1.122130,-1.489063,1.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,-0.370683,-0.827109,-1.200717,1.510988,-1.122130,-0.793223,0.0
2,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.066087,-0.102982,-0.429182,1.510988,0.496561,-1.489063,0.0
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.450328,-0.102982,-1.412377,-0.694638,-0.852348,1.294297,1.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,-0.165430,0.621145,0.321870,0.040570,-1.661693,-1.489063,0.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27817 entries, 0 to 27816
Data columns (total 25 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   cat__Gender_Male                            27817 non-null  float64
 1   cat__Sleep Duration_7-8 hours               27817 non-null  float64
 2   cat__Sleep Duration_Less than 5 hours       27817 non-null  float64
 3   cat__Sleep Duration_More than 8 hours       27817 non-null  float64
 4   cat__Dietary Habits_Moderate                27817 non-null  float64
 5   cat__Dietary Habits_Unhealthy               27817 non-null  float64
 6   cat__Degree_B.Com                           27817 non-null  float64
 7   cat__Degree_B.Ed                            27817 non-null  float64
 8   cat__Degree_B.Tech                          27817 non-null  float64
 9   cat__Degree_BCA                             27817 non-null  float64
 10  cat__Degre

La salida de `df.head()` muestra las primeras 5 filas del conjunto de datos, que contiene varias características categóricas (codificadas one-hot), binarias y numéricas. La variable objetivo, 'remainder__Depression', indica la presencia (1.0) o ausencia (0.0) de depresión.

`df.info()` confirma que hay 27,817 entradas y 25 columnas. Todas las columnas son de tipo `float64` y no tienen valores faltantes, lo que indica que los datos ya han sido preprocesados y limpiados. Esto es crucial para proceder con el entrenamiento del modelo sin pasos adicionales de imputación.

In [5]:
# Definir las características (X) y la variable objetivo (y)
y = df.iloc[:, -1]   # La última columna como objetivo
X = df.drop(df.columns[-1], axis=1) # Todas las columnas excepto la última como características

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de y_train:", y_train.shape)
print("Dimensiones de X_test:", X_test.shape)
print("Dimensiones de y_test:", y_test.shape)

Dimensiones de X_train: (22253, 24)
Dimensiones de y_train: (22253,)
Dimensiones de X_test: (5564, 24)
Dimensiones de y_test: (5564,)


### 2. Preparación de Datos

Antes de entrenar cualquier modelo, el conjunto de datos se divide en características (X) y la variable objetivo (y). Posteriormente, estos se dividen en conjuntos de entrenamiento y prueba para evaluar el rendimiento del modelo con datos no vistos. `test_size=0.2` indica que el 20% de los datos se utilizará para la prueba, y `random_state=42` asegura la reproducibilidad de la división.

In [6]:
print("X_train head:")
display(X_train.head())

X_train head:


,cat__Gender_Male,cat__Sleep Duration_7-8 hours,cat__Sleep Duration_Less than 5 hours,cat__Sleep Duration_More than 8 hours,cat__Dietary Habits_Moderate,cat__Dietary Habits_Unhealthy,cat__Degree_B.Com,cat__Degree_B.Ed,cat__Degree_B.Tech,cat__Degree_BCA,...,cat__Degree_MSc,cat__Degree_Other,bin__Have you ever had suicidal thoughts ?,bin__Family History of Mental Illness,num__Age,num__Academic Pressure,num__CGPA,num__Study Satisfaction,num__Work/Study Hours,num__Financial Stress
20754,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,-1.191694,-1.551237,0.813467,-1.429847,1.036124,-0.097383
8341,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.860834,-0.827109,-1.132440,1.510988,0.766342,1.294297
7582,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.245076,-1.551237,1.298237,0.775779,0.766342,-1.489063
812,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,-1.396947,-0.827109,0.588152,1.510988,0.226779,0.598457
13911,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,-0.370683,-0.827109,-0.811536,1.510988,-0.043003,-0.097383


In [7]:
print("y_train head:")
display(y_train.head())

y_train head:


20754    0.0
8341     1.0
7582     0.0
812      1.0
13911    0.0
Name: remainder__Depression, dtype: float64

Las salidas de `X_train.head()` y `y_train.head()` ofrecen un vistazo a los datos de entrenamiento después de la división. `X_train` contiene el conjunto de características, mientras que `y_train` contiene las etiquetas objetivo correspondientes. Esta verificación asegura que los datos han sido preparados correctamente para el entrenamiento del modelo.

In [8]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy del modelo RandomForestClassifier: {accuracy:.4f}")

Accuracy del modelo RandomForestClassifier: 0.8445


### 3. Entrenamiento del Modelo Base: RandomForestClassifier

Se entrena un `RandomForestClassifier` con hiperparámetros predeterminados como modelo base. La precisión del modelo en el conjunto de prueba se calcula para establecer un punto de referencia para la comparación con modelos con hiperparámetros ajustados.

### Búsqueda de hiperparámetros con GridSearchCV para RandomForestClassifier

Ahora, utilizaremos `GridSearchCV` para una búsqueda exhaustiva de hiperparámetros en el modelo `RandomForestClassifier`. Esto nos permitirá encontrar la combinación óptima para mejorar su rendimiento.

In [9]:
# # ES DEMASIADO COSTOSO COMPUTACIONALMENTE, NO EJECUTA.

# # Definir el modelo RandomForestClassifier
# rf_gs = RandomForestClassifier(random_state=42)

# # Definir la cuadrícula de parámetros para GridSearchCV (ajustada para RandomForestClassifier)
# param_grid_rf_gs = {
#     'n_estimators': [100, 200, 300],  # Número de árboles en el bosque
#     'max_features': ['sqrt', 'log2'], # Número de características a considerar en cada división
#     'max_depth': [10, 20, None],      # Profundidad máxima del árbol (None significa sin límite)
#     'min_samples_split': [2, 5],      # Número mínimo de muestras requeridas para dividir un nodo interno
#     'min_samples_leaf': [1, 2]        # Número mínimo de muestras requeridas en cada nodo hoja
# }

# # Inicializar GridSearchCV para RandomForestClassifier
# grid_search_rf = GridSearchCV(
#     estimator=rf_gs,
#     param_grid=param_grid_rf_gs,
#     cv=5,       # Validación cruzada con 5 folds
#     n_jobs=-1,  # Usar todos los núcleos disponibles
#     verbose=2   # Mostrar detalles del progreso
# )

# # Ajustar GridSearchCV a los datos de entrenamiento
# grid_search_rf.fit(X_train, y_train)

# # Mostrar los mejores parámetros y la mejor puntuación
# print("\nMejores parámetros para RandomForestClassifier (GridSearchCV):", grid_search_rf.best_params_)
# print("Mejor puntuación (accuracy) para RandomForestClassifier (GridSearchCV):", grid_search_rf.best_score_)

# # Evaluar el modelo con los mejores hiperparámetros en el conjunto de prueba
# best_rf_model_gs = grid_search_rf.best_estimator_
# y_pred_rf_gs = best_rf_model_gs.predict(X_test)
# accuracy_rf_gs = accuracy_score(y_test, y_pred_rf_gs)
# print("Precisión de RandomForestClassifier (GridSearchCV) en el conjunto de prueba:", accuracy_rf_gs)

### 4. Ajuste de Hiperparámetros con GridSearchCV

`GridSearchCV` realiza una búsqueda exhaustiva sobre una cuadrícula de parámetros especificada para encontrar la mejor combinación de hiperparámetros para el `RandomForestClassifier`. Aunque es eficaz, este método puede ser computacionalmente costoso, especialmente con un espacio de parámetros y un conjunto de datos grandes. El comentario en el código refleja esto, indicando que fue demasiado costoso ejecutarlo completamente en este entorno.

### Búsqueda de hiperparámetros con RandomizedSearchCV para RandomForestClassifier

Dado que `GridSearchCV` puede ser computacionalmente costoso, utilizaremos `RandomizedSearchCV` para explorar un subconjunto aleatorio del espacio de hiperparámetros, lo que a menudo puede encontrar buenos resultados en menos tiempo.

In [10]:
# Definir el modelo
model_rs = RandomForestClassifier(random_state=41)

# Definir la distribución de parámetros para RandomizedSearchCV
param_dist_rs = {
    'n_estimators': randint(50, 200), # Enteros aleatorios entre 50 y 200
    'max_depth': randint(3, 15), # Enteros aleatorios entre 3 y 15
    'min_samples_split': randint(2, 11),
    'criterion': ['gini', 'entropy']
}

# Inicializar RandomizedSearchCV
# n_iter controla cuántas combinaciones de parámetros se probarán
random_search = RandomizedSearchCV(estimator=model_rs, param_distributions=param_dist_rs,
                                   n_iter=10, cv=3, n_jobs=-1, verbose=2, random_state=42)

# Ajustar RandomizedSearchCV a los datos
random_search.fit(X_train, y_train)

# Mostrar los mejores parámetros y la mejor puntuación
print("\nMejores parámetros para RandomizedSearchCV:", random_search.best_params_)
print("Mejor puntuación (accuracy) para RandomizedSearchCV:", random_search.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits

Mejores parámetros para RandomizedSearchCV: {'criterion': 'entropy', 'max_depth': 14, 'min_samples_split': 10, 'n_estimators': 98}
Mejor puntuación (accuracy) para RandomizedSearchCV: 0.8424482126328529


### 5. Ajuste de Hiperparámetros con RandomizedSearchCV

Para abordar el costo computacional de `GridSearchCV`, se emplea `RandomizedSearchCV`. Este método muestrea un número fijo de configuraciones de parámetros de distribuciones especificadas, ofreciendo una forma más eficiente de explorar el espacio de hiperparámetros. `n_iter` controla el número de combinaciones de parámetros muestreadas, y `cv` especifica el número de divisiones de validación cruzada.

La búsqueda identifica los hiperparámetros óptimos, que luego se utilizan para entrenar un `RandomForestClassifier` refinado.

In [11]:
# Mejores parámetros de RandomizedSearchCV para RandomForestClassifier
print("Mejores parámetros de RandomForestClassifier (RandomizedSearchCV):", random_search.best_params_)

# Entrenar un modelo final con los mejores parámetros de RandomizedSearchCV
best_rf_random_model = RandomForestClassifier(**random_search.best_params_, random_state=42)
best_rf_random_model.fit(X_train, y_train)
y_pred_random = best_rf_random_model.predict(X_test)
accuracy_random = accuracy_score(y_test, y_pred_random)
print(f"\nPrecisión del modelo final (RandomizedSearchCV) en el conjunto de prueba: {accuracy_random:.4f}")

# Comparación de rendimientos
print("\n--- Comparación de Rendimiento ---")
print(f"Accuracy por defecto: {accuracy:.4f}")
print(f"Accuracy RandomizedSearchCV: {accuracy_random:.4f}")

if accuracy_random > accuracy:
    print("RandomizedSearchCV obtuvo una mejor precisión.")
elif accuracy > accuracy_random:
    print("El modelo por defecto obtuvo una mejor precisión.")
else:
    print("Ambos métodos obtuvieron la misma precisión.")

Mejores parámetros de RandomForestClassifier (RandomizedSearchCV): {'criterion': 'entropy', 'max_depth': 14, 'min_samples_split': 10, 'n_estimators': 98}

Precisión del modelo final (RandomizedSearchCV) en el conjunto de prueba: 0.8453

--- Comparación de Rendimiento ---
Accuracy por defecto: 0.8445
Accuracy RandomizedSearchCV: 0.8453
RandomizedSearchCV obtuvo una mejor precisión.


### 6. Evaluación y Comparación del Modelo Final

Después de identificar los mejores hiperparámetros utilizando `RandomizedSearchCV`, se entrena un modelo `RandomForestClassifier` final con estas configuraciones optimizadas. La precisión de este modelo se compara con la del modelo base. En este caso, `RandomizedSearchCV` resultó en una precisión ligeramente mejorada, lo que demuestra el beneficio del ajuste de hiperparámetros.

### Cómo usar el hiperparámetro `best_rf_random_model`

Una vez que hemos entrenado `best_rf_random_model` con los mejores hiperparámetros encontrados por `RandomizedSearchCV`, este modelo está listo para realizar predicciones sobre nuevos datos.

**Recuerda:** Los nuevos datos sobre los que quieras predecir deben tener el mismo formato (mismas columnas y preprocesamiento, por ejemplo, escalado si se aplicó) que los datos de entrenamiento (`X_train`).

Aquí te mostramos cómo puedes usarlo para hacer predicciones:

In [12]:
# Acceder a los mejores parámetros
best_hyperparameters = random_search.best_params_

print("\nLos mejores hiperparámetros encontrados por RandomizedSearchCV son:")
for param, value in best_hyperparameters.items():
    print(f"  {param}: {value}")

output_filename = '..\outputs\params\\best_random_forest_hyperparameters.json'
with open(output_filename, 'w') as f:
    json.dump(best_hyperparameters, f, indent=4)

print(f"\nLos mejores hiperparámetros también se han guardado en '{output_filename}'")


Los mejores hiperparámetros encontrados por RandomizedSearchCV son:
  criterion: entropy
  max_depth: 14
  min_samples_split: 10
  n_estimators: 98

Los mejores hiperparámetros también se han guardado en '..\outputs\params\best_random_forest_hyperparameters.json'


### 7. Exportación de los Mejores Hiperparámetros

Finalmente, los mejores hiperparámetros encontrados por `RandomizedSearchCV` se extraen y se guardan en un archivo JSON. Esto permite una fácil reproducibilidad y despliegue del modelo optimizado sin necesidad de volver a ejecutar la búsqueda de hiperparámetros cada vez.